# K2_104 — Single-Stage FAVAR Flow (Colab GPU 학습)

Base 모델 (K2_pure_ppbond) 의 condition 변형 실험.

**구조**
- IN cond (4ch × 104w): `excess_liq_yoy`, `tbill_wr`, `tbill_26w_lag`, `excess_liq_26w_lag`
- IN past target (1ch × past 52w): `sp_return`
- OUT future target (1ch × future 52w): `sp_return`
- Architecture: `MultiStepFAVARFlow K=2, d_model=64, n_heads=4, n_layers=2` (NLL only)

**준비**
1. 이 폴더 (`k2_104/`) 통째로 Google Drive `MyDrive/Colab Notebooks/` 아래 업로드.
2. Runtime → Change runtime type → **T4 또는 A100 GPU**.
3. 셀 순서대로 실행.

## 1. Drive mount + 작업 디렉토리 이동

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKDIR = '/content/drive/MyDrive/Colab Notebooks/k2_104'
os.chdir(WORKDIR)
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))

## 2. GPU 확인

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
    print('VRAM total     :', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 3. 학습 (단일 시드, 빠른 검증)

T4 예상 시간: 약 5~10분 (60 epoch, early stop 가능).

In [ ]:
!python train.py --seeds 42 2>&1 | tee result/K2_104_seed42_run.log

## 4. 결과 확인 — 학습 곡선 + best NLL

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('result/K2_104_seed42_trainlog.csv')
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(log['epoch'], log['train'], label='train')
ax.plot(log['epoch'], log['val'],   label='val')
ax.plot(log['epoch'], log['test'],  label='test')
ax.set_title('K2_104  NLL/step/channel  (sp_return future 52w)')
ax.set_xlabel('epoch'); ax.set_ylabel('NLL')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

with open('result/K2_104_seed42_summary.json') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

## 5. (선택) 멀티 시드 (n=5) 통계 비교

T4에서 약 30~50분.

In [ ]:
# !python train.py --seeds 42 123 777 0 99 2>&1 | tee result/K2_104_multi_run.log